# Clasificación de textos de amazan con transformer utilizando tecnicas de fine tuning

## Librerias

In [ ]:
import torch
import numpy as np
import random
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)


c:\Users\Alumno\Documents\clasificacion_opiniones_amazon\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Verificar que se trabaja con gpu

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print( device )

cuda


## Importar dataset de huggie faces

In [2]:
dataset = load_dataset("SetFit/amazon_reviews_multi_es")

c:\Users\Alumno\Documents\clasificacion_opiniones_amazon\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Alumno\.cache\huggingface\hub\datasets--SetFit--amazon_reviews_multi_es. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Repo card metadata block was not found. Setting CardData to empty.
Gener

## Analisis del dataset

In [3]:
import collections

In [4]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 5000
    })
})


In [5]:
print("Numero de instancias del conjunto entrenamiento")
print(collections.Counter(dataset["train"]["label"]))

Numero de instancias del conjunto entrenamiento
Counter({0: 40000, 1: 40000, 2: 40000, 3: 40000, 4: 40000})


In [6]:
print("Numero de instancias del conjunto validación")
print(collections.Counter(dataset["validation"]["label"]))

Numero de instancias del conjunto validación
Counter({0: 1000, 1: 1000, 2: 1000, 3: 1000, 4: 1000})


In [7]:
print("Numero de instancias del conjunto prueba")
print(collections.Counter(dataset["test"]["label"]))

Numero de instancias del conjunto prueba
Counter({0: 1000, 1: 1000, 2: 1000, 3: 1000, 4: 1000})


In [8]:
textos = dataset["train"]["text"]
palabras_maximas = 0
palabras_minimas = np.inf
for texto in textos:
    numero_palabras = len( texto.split() )
    if palabras_maximas < numero_palabras:
        palabras_maximas = numero_palabras
    if palabras_minimas > numero_palabras:
        palabras_minimas = numero_palabras

print( f"Número de palabras maximas en un parrafo: {palabras_maximas}" )
print( f"Número de palabras minimas en un parrafo: {palabras_minimas}" )

Número de palabras maximas en un parrafo: 551
Número de palabras minimas en un parrafo: 2


## Cargar tokenizador

In [9]:
model_name = "xlm-roberta-large"
tokenizador = AutoTokenizer.from_pretrained(model_name)

c:\Users\Alumno\Documents\clasificacion_opiniones_amazon\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Alumno\.cache\huggingface\hub\models--xlm-roberta-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


### Tokenizar

In [ ]:
def tokenizar(ejemplo):
    return tokenizador( ejemplo["text"], truncation=True, padding=True, max_length=512 )

In [12]:
datos_tokenizados = dataset.map(tokenizar, batched=True)

Map: 100%|██████████| 5000/5000 [00:00<00:00, 19746.54 examples/s]


In [13]:
datos_tokenizados = datos_tokenizados.remove_columns(["text", "label_text"])
datos_tokenizados.set_format("torch")

In [14]:
print( datos_tokenizados )

DatasetDict({
    train: Dataset({
        features: ['id', 'label', 'input_ids', 'attention_mask'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['id', 'label', 'input_ids', 'attention_mask'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['id', 'label', 'input_ids', 'attention_mask'],
        num_rows: 5000
    })
})


## Padding de los datos tokenizados

In [15]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizador)
modelo = AutoModelForSequenceClassification.from_pretrained( model_name, num_labels=5 ) # Se adapta al modelo para realizar clasificación

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 13616.51it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Tecnica LoRA

In [16]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16, # Es el doble de r
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

modelo = get_peft_model(modelo, lora_config)

modelo.print_trainable_parameters()

trainable params: 1,841,157 || all params: 561,736,714 || trainable%: 0.3278


## Calcular métricas detalladas de rendimiento en modelos de clasificación

In [17]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

## Definir y configurar todos los hiperparámetros

In [ ]:
training_args = TrainingArguments(
    output_dir="modelo-ajustado",

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    #metric_for_best_model="eval_f1_macro",
    #greater_is_better=True,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    optim="adamw_torch",

    num_train_epochs=5,
    learning_rate=2e-4,
    weight_decay=0.01,

    logging_strategy="epoch",
    save_total_limit=2,
    report_to="none"
)

## Sección de Entrenamiento

In [ ]:
trainer = Trainer(
    model=modelo,
    args=training_args,
    train_dataset=datos_tokenizados["train"].shuffle(seed=42).select(range(2000)),
    eval_dataset=datos_tokenizados["validation"].shuffle(seed=42).select(range(400)),
    data_collator=data_collator,

    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

In [ ]:
trainer.train()

## Sección de Pruebas

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


logs = pd.DataFrame(trainer.state.log_history)

train_loss = logs[logs["loss"].notna()].copy()
train_loss["epoch_int"] = train_loss["epoch"].astype(int)
train_loss_epoch = train_loss.groupby("epoch_int")["loss"].mean()

eval_loss = logs[logs["eval_loss"].notna()].copy()
eval_loss["epoch_int"] = eval_loss["epoch"].astype(int)
eval_loss_epoch = eval_loss.groupby("epoch_int")["eval_loss"].mean()

best_epoch = eval_loss_epoch.idxmin()
best_value = eval_loss_epoch.min()

plt.figure(figsize=(8,5))

plt.plot(train_loss_epoch.index, train_loss_epoch.values, marker="o", label="Train Loss")
plt.plot(eval_loss_epoch.index, eval_loss_epoch.values, marker="o", label="Validation Loss")

plt.scatter(best_epoch, best_value)
plt.text(best_epoch, best_value, "  Mejor modelo", fontsize=10)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Validation Loss (por epoch REAL)")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
best_model_path = trainer.state.best_model_checkpoint
print("Mejor modelo en:", best_model_path)

modelo_mejor = AutoModelForSequenceClassification.from_pretrained(best_model_path)

eval_args = TrainingArguments(
    output_dir="eval",
    per_device_eval_batch_size=8
)

trainer_best = Trainer(
    model=modelo_mejor,
    args=eval_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

resultados_test = trainer_best.evaluate(datos_tokenizados["test"])
print("\n=== Evaluación en TEST ===")
print(f"Precision:        {resultados_test['eval_precision']:.4f}")
print(f"Recall:           {resultados_test['eval_recall']:.4f}")
print(f"F1-score:         {resultados_test['eval_f1']:.4f}")